In [1]:
import plotly.express as px
import duckdb
import pandas as pd
import numpy as np

master_sales_cleaned = '../CSV_read_only/APPLE_2024_2025_PIPE_RESULT.parquet'
master_traffic_cleaned = '../CSV_read_only/APPLE_2024_2026_FAKE_TRAFFIC.parquet'


duckdb.query(
    f"""--sql
    ;
    """
)

In [ ]:
df_ready = duckdb.query(
    f"""--sql
    SELECT * 
    FROM '{master_sales_cleaned}'
    ;
    """)

df_traffic = duckdb.query(
    f"""--sql
    SELECT * 
    FROM '{master_traffic_cleaned}'
    ;
    """)

df_master = duckdb.query(
    f"""--sql
    SELECT 
    * EXCLUDE(Period, new_Traffic, Event_name),
    new_Traffic as date_traffic,
    Event_name as event_name 
    FROM df_ready s
    LEFT JOIN df_traffic t
    ON s.date = t.Period
    ORDER BY date ASC
    ;
    """).df().convert_dtypes()
# XXX 
# HACK 
# FIXME 
# NOTE
df_master

In [3]:
import pandas as pd
import duckdb

def join_sales_traffic(df: any, traffic_path: str) -> pd.DataFrame:
    # 1. Xử lý đầu vào df
    if isinstance(df, str):
        # Nếu df là đường dẫn (string), ta dùng hàm read_parquet hoặc read_csv của DuckDB
        # Ở đây ta chỉ cần giữ nguyên đường dẫn để đưa vào SQL với dấu nháy đơn
        df_input = f"'{df}'"
    elif isinstance(df, pd.DataFrame):
        # Nếu là DataFrame, DuckDB sẽ nhìn thấy biến 'df' trực tiếp
        df_input = "df"
    else:
        raise TypeError("df must be either a string path or a pandas DataFrame")

    query = f"""--sql
        SELECT 
        * EXCLUDE (Period, new_Traffic, Event_name),
        new_Traffic as date_traffic,
        Event_name as event_name 
        FROM {df_input} s
        LEFT JOIN '{traffic_path}' t
        ON s.date = t.Period
        ORDER BY date ASC;
    """
    
    return duckdb.query(query).df().convert_dtypes()

df_master = join_sales_traffic(master_sales_cleaned, master_traffic_cleaned)
df_master

,date,invoice,staff,sku,imei_sn,cat,detail_sub_lob,product_name,price,qty,...,card,qr_code,time,id,name,email,color,memory_size,date_traffic,event_name
0,2024-04-01,10000,STAFF_13,3RD-CVZFO-M77LA,unknown,3RD ACC,Powerbank,DISPLAY 15W WIRELESS POWER BANK 10000,1529000,1,...,1529000.0,0.0,1900-01-01 22:35:00,<NA>,<NA>,<NA>,<NA>,<NA>,280,<NA>
1,2024-04-01,10001,STAFF_13,APP-ARM4X-CPM7Q,unknown,ACCESSORIES (APPLE),Charger,20W USB-C POWER ADAPTER,590000,1,...,0.0,590000.0,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,280,<NA>
2,2024-04-01,10002,STAFF_13,APP-ARMDO-UGTLQ,unknown,ACCESSORIES (APPLE),Charger,USB-C TO APPLE PENCIL ADAPTER,390000,1,...,0.0,0.0,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,280,<NA>
3,2024-04-01,10003,STAFF_04,3RD-CVZFO-MPQMQ,unknown,3RD ACC,Cable,MFI SYNC CHARGE C TO LIGHTNING CAB 10M,529000,1,...,0.0,529000.0,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,280,<NA>
4,2024-04-01,10004,STAFF_04,3RD-DTUAZ-34ZSQ,unknown,3RD ACC,Normal,MIẾNG DÁN CƯỜNG LỰC MIPOW KINGBULL HD PREMIUM-...,319000,1,...,319000.0,0.0,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,280,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17756,2025-12-31,21185,STAFF_11,APP-ARM4X-2AYGQ,unknown,ACCESSORIES (APPLE),Iphone 17 Pro Max Case,IPHONE 17 PRO MAX CASE CLEAR,1490000,1,...,1490000.0,0.0,1900-01-01 21:18:00,<NA>,<NA>,<NA>,Clear,<NA>,480,<NA>
17757,2025-12-31,21185,STAFF_11,3RD-DTYFM-OWMKQ,unknown,3RD ACC,Front Screen,JCP4538,489000,1,...,489000.0,0.0,1900-01-01 21:18:00,<NA>,<NA>,<NA>,<NA>,<NA>,480,<NA>
17758,2025-12-31,21186,STAFF_11,3RD-DTUAZ-37I3A,unknown,3RD ACC,Case,CASE ORANGE,599000,1,...,599000.0,0.0,1900-01-01 17:29:00,<NA>,<NA>,<NA>,Orange,<NA>,480,<NA>
17759,2025-12-31,21186,STAFF_11,3RD-DTUAZ-37ASQ,unknown,3RD ACC,Front Screen,BJ712-BK,489000,1,...,489000.0,0.0,1900-01-01 17:48:00,<NA>,<NA>,<NA>,<NA>,<NA>,480,<NA>


In [2]:
import streamlit as st
kpis = [
    {"label": "Total Revenue", "value": "$7,716,281", "delta": "+56.6%"},
    {"label": "Total Transaction", "value": "11.8%", "delta": "+0.5%"},
    {"label": "Average Ticket Value", "value": "15,277", "delta": "-55.6%"},
    {"label": "Unit Per Transaction", "value": "$505", "delta": "+0.1%"}
]

dict_1 = {'key': 234}
dict_1

{'key': 234}

In [3]:
def bf_fill(df: pd.DataFrame, _anchor, _target_cols)-> pd.DataFrame:
    """
    ### Fill 1 hoặc nhiều cột bằng anchor columns
    """
    df[_target_cols] = df[_target_cols].replace(r'^\s*$', np.nan, regex=True)
    df[_target_cols] = df.groupby(_anchor)[_target_cols].transform('bfill')
    df[_target_cols] = df.groupby(_anchor)[_target_cols].transform('ffill')
    return df

def get_top_report(df, group_cols, target='revenue', n=10):
    return (df.groupby(group_cols, as_index=False)
            .agg({'qty': 'sum', 'revenue': 'sum', 'invoice': 'nunique'})
            .nlargest(n, target)
            .reset_index(drop=True))

# Sử dụng cực kỳ gọn:
cat_report = get_top_report(df_master, ['cat'])
sku_report = get_top_report(df_master, ['cat', 'sku'], target='qty')

In [ ]:
_cust_anc = 'invoice'
_cust_cols = ['id', 'name', 'email']
df_master = df_master.pipe(bf_fill, _anchor=_cust_anc, _target_cols=_cust_cols)

top_seller_qty = (
    df_master.groupby(['cat', 'sku', 'detail_sub_lob', 'product_name'], as_index=False)
    .agg({'qty': 'sum', 'revenue': 'sum', 'invoice': 'nunique'})
    .sort_values(by='qty', ascending=False, ignore_index=True)
)

top_seller_rev = (
    df_master.groupby(['cat', 'product_name'], as_index=False)
    .agg({'qty': 'sum', 'revenue': 'sum', 'invoice': 'nunique'})
    .sort_values(by='revenue', ascending=False, ignore_index=True)
)

top_seller_cat = (
    df_master.groupby(['cat'], as_index=False)
    .agg({'qty': 'sum', 'revenue': 'sum', 'invoice': 'nunique'})
    .sort_values(by='revenue', ascending=False, ignore_index=True)
).query("revenue > 0")

top_iphone = (
    df_master.query("cat == 'IPHONE'")
    .groupby(['product_name'], as_index=False)
    .agg({'qty': 'sum', 'revenue': 'sum'})
    .sort_values(by='qty', ascending=False)
)


cat_contribution = (
    df_master.groupby(['cat'], as_index=False)
    .agg({'revenue': 'sum'})
    .assign(pct_contribution = lambda x: (x['revenue'] / df_master['revenue'].sum()) * 100)
    .sort_values(by='revenue', ascending=False)
)

potential_products = (
    top_seller_qty.query("qty < qty.median()") # Bán ít hơn mức trung bình
    .nlargest(10, 'revenue')                   # Nhưng doanh thu cao nhất nhóm đó
)

trend_cat = (
    df_master.groupby([pd.Grouper(key='date', freq='M'), 'cat'])['revenue']
    .sum()
    .unstack() # Chuyển Category lên làm cột để vẽ biểu đồ đường
    .fillna(0)
)

trend_cat.head(15)

In [ ]:
import plotly.graph_objects as go
labels_cat = top_seller_cat['cat'].tolist()
values_raw = top_seller_cat['revenue'].tolist()

fig = px.pie()

fig.add_pie(
    labels=labels_cat,
    values=values_raw,
    pull=None,
    hole=0.42,
    sort=True,
    direction='clockwise',
    opacity=1,
    marker=dict(
        colors=['#2C528C', '#4281AF', '#78C5EF', '#A0D8F1', '#D1EAF5', '#ECECEC'],
        line=dict(color='#FFFFFF', width=2)
    ),
    textposition='outside',
    textinfo='label+percent',
    insidetextorientation='horizontal',
    hoverinfo='label+value+percent',
    hovertemplate="<b>%{label}</b><br>Doanh thu: %{value:,.0f} VNĐ<br>Tỉ trọng: %{percent}<extra></extra>"
)

# 3. Thêm Annotations và Layout theo phong cách bạn yêu cầu
fig.update_layout(
    title='<b>Cơ cấu Doanh thu theo Danh mục</b> <span style="font-size:14px; color:#888;">(Báo cáo 2025)</span>',
    template="plotly_dark",
    font_family='Segoe UI Semibold',
    font_color="#2C528C",
    title_font_size=24,
    
    legend=dict(
        orientation="v",
        yanchor="middle", y=0.5,
        xanchor="center", x=0.9,
        font=dict(size=12, color="#86868b")
    ),
    
    margin=dict(t=120, b=60, l=60, r=60),
    
    # annotations=[
    #     dict(
    #         text='<b>TOTAL</b><br>REVENUE', 
    #         x=0.5, y=0.5, 
    #         showarrow=False,
    #         font=dict(size=16, color="#2C528C")
    #     ),

    #     dict(
    #         text="<b>Insight:</b> Nhóm IPHONE đóng góp hơn 50%.",
    #         align='left', showarrow=False,
    #         x=1.1, y=0.1, xref="paper", yref="paper",
    #         font=dict(size=11, color="gray")
    #     )
    # ]
)


### Analysis_SQL

In [4]:
# ------ config ------
source_df         = 'df_master'
date_col          = 'date'
time_col          = 'time'
qty_col           = 'qty'
rev_column        = 'revenue'        
event_start_date  = '2025-09-18'    
analysis_end_date = '2025-12-31' 
rolling_window    = 6               
# ----------------------

# astype cols SQL style
duck_master = duckdb.sql(f"""--sql
    SELECT *
    REPLACE (
        {qty_col}::INT32 as qty,
        {date_col}::DATE as date,
        {time_col}::TIMESTAMP as time
    )
    FROM {source_df};
""")
# Phải groupby cho mỗi ngày thành 1 dòng trước 
# vì AVG() nó không quan tâm 1 ngày có bao nhiêu dòng, 
# nếu 1 ngày càng nhiều dòng thì số AVG càng nhỏ và chả phản ánh đúng cái gì cả

duck_date_rev = duckdb.sql(f"""--sql
    WITH pre_cal AS (
        SELECT
            date as Date,
            SUM({rev_column}) as Revenue_by_Date                                            -- #!!!
        FROM duck_master
        GROUP BY date
        ORDER BY date ASC
    ),

    raw_cal AS (
        SELECT
            Date,
            Revenue_by_Date,
            ROUND(AVG(Revenue_by_Date) OVER(
                ORDER BY Date ROWS BETWEEN {rolling_window} PRECEDING AND CURRENT ROW
            ), 0) as Rolling_7Day            
        FROM pre_cal
    ),

    sqrt_abc AS (
        SELECT
            *,
            ROUND(SQRT(Revenue_by_Date), 0) as sqrt_revenue, 
            ROUND(SQRT(Rolling_7Day), 0)    as sqrt_rolling,
            CASE WHEN Date >= '{event_start_date}' AND Date <= '{analysis_end_date}' 
                 THEN ROUND(SQRT(Rolling_7Day), 0) ELSE NULL END as sqrt_event_launching,
            CASE WHEN Date >= '{event_start_date}' AND Date <= '{analysis_end_date}' 
                 THEN 1 ELSE 0 END as is_after_event
        FROM raw_cal
    ),

    mean_abc AS (
        SELECT 
            *,  
            AVG(CASE WHEN is_after_event = 0 THEN Revenue_by_Date ELSE NULL END) OVER() as rbm,
            AVG(CASE WHEN is_after_event = 1 THEN Revenue_by_Date ELSE NULL END) OVER() as ram
        FROM sqrt_abc
    )

    SELECT 
        *,
        CASE WHEN is_after_event = 0 THEN rbm ELSE NULL END as raw_before_event_mean,
        CASE WHEN is_after_event = 1 THEN ram ELSE NULL END as raw_after_event_mean,
        CASE WHEN is_after_event = 0 THEN SQRT(rbm) ELSE NULL END as sqrt_before_event_mean,
        CASE WHEN is_after_event = 1 THEN SQRT(ram) ELSE NULL END as sqrt_after_event_mean,
        (ram / rbm - 1) * 100 AS event_increase
    FROM mean_abc;
""").df()

if True: 
    number_cols = duck_date_rev.select_dtypes(include='number').columns
    duck_date_rev[number_cols] = duck_date_rev[number_cols].round(2).convert_dtypes()

duck_date_rev

,Date,Revenue_by_Date,Rolling_7Day,sqrt_revenue,sqrt_rolling,sqrt_event_launching,is_after_event,rbm,ram,raw_before_event_mean,raw_after_event_mean,sqrt_before_event_mean,sqrt_after_event_mean,event_increase
0,2024-04-01,15256000,15256000,3906,3906,<NA>,0,175909829.33,392503749.52,175909829.33,<NA>,13263.1,<NA>,123.13
1,2024-04-02,64073000,39664500,8005,6298,<NA>,0,175909829.33,392503749.52,175909829.33,<NA>,13263.1,<NA>,123.13
2,2024-04-03,43466000,40931667,6593,6398,<NA>,0,175909829.33,392503749.52,175909829.33,<NA>,13263.1,<NA>,123.13
3,2024-04-04,82987000,51445500,9110,7173,<NA>,0,175909829.33,392503749.52,175909829.33,<NA>,13263.1,<NA>,123.13
4,2024-04-05,75511000,56258600,8690,7501,<NA>,0,175909829.33,392503749.52,175909829.33,<NA>,13263.1,<NA>,123.13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
632,2025-12-27,356860000,378815286,18891,19463,19463,1,175909829.33,392503749.52,<NA>,392503749.52,<NA>,19811.71,123.13
633,2025-12-28,505800000,403543571,22490,20088,20088,1,175909829.33,392503749.52,<NA>,392503749.52,<NA>,19811.71,123.13
634,2025-12-29,386822000,413280286,19668,20329,20329,1,175909829.33,392503749.52,<NA>,392503749.52,<NA>,19811.71,123.13
635,2025-12-30,206251000,390383000,14361,19758,19758,1,175909829.33,392503749.52,<NA>,392503749.52,<NA>,19811.71,123.13


### First Chart
Có thể add vào bất cứ chart nào\
fig.add_scatter( )\
fig.add_bar( )\
fig.add_histogram( )\
fig.add_vline( )\
fig.add_hline( )\
fig.add_annotation( )\
fig.update_traces( )\
fig.update_xaxes( )\
fig.update_yaxes( )\
fig.update_layout( )

In [ ]:
tick_vals_raw = [
    0, 200_000_000, 600_000_000, 
    1_200_000_000,  1_800_000_000
    ]
y_max_sqrt = (max(tick_vals_raw) + 4e8) **0.5
tick_text_raw = ['0', '200 Tr', '600 Tr', '1,2 Tỉ', '1,8 Tỉ']
tick_vals_sqrt = np.sqrt(tick_vals_raw).tolist()

if True: #! Params
    #todo Không thể gán value cho vế dùng function/method
    x_raw               = 'Date'
    y_sqrt              = ['sqrt_revenue', 'sqrt_rolling']
    y_raw               = ['Revenue_by_Date', 'Rolling_7Day']
    legend_a            = 'Daily Revenue'
    legend_b            = '7D Trend'

    # title & range
    x_title           = 'Tháng'
    y_title           = 'VNĐ'
    x_range           = ["2025-01-01", "2025-12-31"]
    y_range           = [0, y_max_sqrt]

    # Event & Mean columns
    sqrt_event_end_year = 'sqrt_event_launching' 
    raw_b_mean          = 'raw_before_event_mean'
    raw_a_mean          = 'raw_after_event_mean'
    sqrt_b_mean         = 'sqrt_before_event_mean'
    sqrt_a_mean         = 'sqrt_after_event_mean'
    event_increase      = duck_date_rev.loc[0, 'event_increase']


fig = px.line(duck_date_rev, 
    title = '<b>AAR Store: Doanh thu 2025</b> <span style="font-size:14px; color:#888;">(VNĐ - sqrt scale)</span>',
    x = x_raw, 
    y = y_sqrt,                                       # Truyền list tên cột vào đây (cột để vẽ, có thể đã Square hoặc log)
    custom_data = y_raw, # Custom_data = Values Raw dùng để show kết quả hover. 
    labels = {'variable': '<b>Metric</b>'},            # Để đặt tên trục/ legend
    hover_name = x_raw,
    # color = None,
    color='variable',
    animation_frame=None, #! Test
    render_mode = 'svg'
    # width=1400, height=550
)

# Rolling 7D
fig.update_traces(
    selector={'name': y_sqrt[1]},
    name = legend_b,
    showlegend = True,
    line_color = "#4281AF", 
    line = dict(
        dash = 'solid',
        width = 2.8,
        shape = 'spline',
        smoothing = 1.3
        ),
    opacity = 1,
    fill = 'tozeroy', # Đổ màu từ đường line xuống trục 0
    fillcolor = 'rgba(120, 197, 239, 0.1)', 
    hovertemplate = "<b>Tuần: %{x|%W}</b><br>Trung bình: %{customdata[1]:,.0f} VNĐ<extra></extra>",
    # visible = 'legendonly'
)
# Daily Revenue
fig.update_traces(
    selector={'name': y_sqrt[0]},
    name = legend_a,
    showlegend = True,
    # line_color="#5EAEFF", 
    line_color = "#367050", 
    line = dict(
        width = 0.8, 
        shape = 'hv', 
        smoothing = 1.3
        ),
    opacity = 0.4,
    hovertemplate = "<b>Ngày: %{x|%d-%m-%Y}</b><br>Doanh thu: %{customdata[0]:,.0f} VNĐ<extra></extra>"
    # fill='tozeroy', # Đổ màu từ đường line xuống trục 0
    # fillcolor='rgba(135, 206, 250, 0.18)', # Màu xanh nhạt với độ trong suốt 20%
)
# Event_launch -> End year
fig.add_scatter(
    line_color = "#4281AF",
    x = duck_date_rev[x_raw], 
    y = duck_date_rev[sqrt_event_end_year],
    fill = 'tozeroy',
    # fillcolor='rgba(255, 167, 0, 0.2)',
    fillcolor = 'rgba(120, 200, 40, 0.2)',
    line = dict(width = 2.7,       # Ẩn = 0
                shape = 'spline', 
                smoothing = 1.3),    
    name = '',                      #! Ẩn
    showlegend = False,             # Giấu khỏi bảng chú thích (Legend)
    hoverinfo = 'skip'              # Rê chuột vào không hiện gì cả
)
# Mean line before_npi
fig.add_scatter(
    x=duck_date_rev[x_raw], 
    y=duck_date_rev[sqrt_b_mean], 
    mode='lines', 
    line=dict(color="#467797", width=1.7, dash='longdashdot'), # Chỉnh trực tiếp ở đây là chắc ăn nhất
        # BỘ BA "VÔ HÌNH":
    showlegend=False,        # 1. Giấu khỏi bảng chú thích (Legend)
    hoverinfo='skip',        # 2. Rê chuột vào không hiện gì cả
    name=''                  # 3. Để tên trống để tránh hiện lỗi nếu lỡ hover trúng
)
# Mean line after_npi
fig.add_scatter(
    x=duck_date_rev[x_raw], 
    y=duck_date_rev[sqrt_a_mean], 
    mode='lines', 
    line=dict(color="#4EA073", width=1.7, dash='longdashdot'), # Chỉnh trực tiếp ở đây là chắc ăn nhất
        # BỘ BA "VÔ HÌNH":
    showlegend=False,        # 1. Giấu khỏi bảng chú thích (Legend)
    hoverinfo='skip',        # 2. Rê chuột vào không hiện gì cả
    name=''                  # 3. Để tên trống để tránh hiện lỗi nếu lỡ hover trúng
)


fig.update_yaxes(
    tickmode = "array",
    tickvals = tick_vals_sqrt, # Vị trí đặt vạch chia (trên thang đo sqrt)
    ticktext = tick_text_raw, # Chữ hiển thị tại vạch đó
    tickprefix = None,
    ticksuffix = '    ',
    automargin = True,
    title = y_title,
    range = y_range,
)
fig.update_xaxes(
    ticklabelmode = "period",
    title_text = x_title,
    range = x_range, 
    title_font = dict(size=12),
    dtick = "M1",      # Có thể là 3 tháng, 1 ngày, 1 giờ...
    tickformat = "%b %y", # %b = chuẩn ISO cho tháng dạng Jan,...
    showgrid = True,
    gridwidth = 1, 
    gridcolor = 'white',
    # x_line = off
    showline = False, 
)
# Thêm đường Trần (y = max(y))
fig.add_hline(y = y_max_sqrt, 
              line_dash="solid", 
              line_color="#D3D3D3", 
              line_width=1)
# Thêm đường Sàn (ví dụ 0 VNĐ)
fig.add_hline(y=0, 
              line_dash="solid", 
              line_color="#D3D3D3", 
              line_width=0.5)
# Thêm Event Line
fig.add_vline(
    x = event_start_date, 
    line_width = 3.1, 
    line_dash = "longdash", 
    line_color = "#E8732A",
    layer = 'above'
)
fig.add_annotation(
    x = event_start_date, 
    y = 1.015, yref = "paper",
    text = "<b>🚀 iPhone 17 NPI</b><br>September 19, 2025.",
    align = 'left',
    showarrow = False,
    font = dict(
        family="Segoe UI Semibold",
        size=10,         
        color="#F28136",   
    ),
    xshift = -5,            # Đẩy chữ sang phải 5px để không dính sát vào đường kẻ
    yanchor = "bottom",
    xanchor = "left"
)
fig.add_annotation(
    text=f"<b>▲ {event_increase:.1f}%</b><br>Vượt mức trung bình trước NPI<br>duy trì đến hết năm 2025.",
    x='2025-10-21',  #! x theo df
    y=0.72,          #! y theo paper
    yref="paper",    # Bắt buộc có dòng này để y= có ý nghĩa
    showarrow=False,
    align="left",
    xanchor="left", 
    yanchor="top",  
    font=dict(size=10, color="gray")          
)

#! Theme | Legend | Margin | X-Y show title | Grid_color | Font | Hover_mode.
fig.update_layout(
    template="plotly_white",
    #! Legend
    legend=dict(
        orientation="h",        # Legend nằm ngang
        yanchor="top", y=1.175, 
        xanchor="center", x=0.5,
        font=dict(size=12, color="#86868b")
        ),
    #! Lề
    margin=dict(
        b=40,
        t=80, 
        l=40,  
        r=40),

    #! Font
    font_family='Segoe UI Semibold',
    font_color="#2C528C",
    title_font_size=26,

    #! X-axis
    xaxis=dict(
        title=None,             # Bỏ
        gridcolor="#D3D3D3",
        gridwidth = 0.2,
        showgrid=False,
        tickfont=dict(size=12),
        title_font=dict(size=12)
    ),
    #! Y-axis
    yaxis=dict(
        gridcolor = "#ECECEC",
        gridwidth = 0.1,
        showgrid = False,
        tickfont = dict(size=12),
        title_font = dict(size=12)
    ),
    hovermode="x unified",
    transition_duration=800,
    hoverlabel=dict(bgcolor="rgba(255, 255, 255, 0.9)", font_size=13),

# Thêm Button vào trong fig.update_layout

)
# fig.update_xaxes(rangeslider_visible=True)
fig.show()